# Prompt Engineering Techniques

This notebook demonstrates a variety of prompt engineering techniques that can be used to improve the quality and reliability of responses from large language models (LLMs).

## Table of Contents
1. [Setup](#1-setup)
2. [Zero-Shot Prompting](#2-zero-shot-prompting)
3. [Few-Shot Prompting](#3-few-shot-prompting)
4. [Chain-of-Thought (CoT) Prompting](#4-chain-of-thought-cot-prompting)
5. [Role Prompting](#5-role-prompting)
6. [Self-Consistency](#6-self-consistency)
7. [Prompt Chaining](#7-prompt-chaining)
8. [Tree of Thoughts (ToT)](#8-tree-of-thoughts-tot)
9. [ReAct Prompting](#9-react-prompting)
10. [Generated Knowledge Prompting](#10-generated-knowledge-prompting)
11. [Directional Stimulus Prompting](#11-directional-stimulus-prompting)

---

## 1. Setup

Install and configure the OpenAI client. Store your API key in a `.env` file as `OPENAI_API_KEY=sk-...`.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install openai python-dotenv

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"  # Change to your preferred model

def chat(messages, temperature=0, model=MODEL):
    """Helper that calls the Chat Completions API and returns the assistant reply."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

def ask(prompt, system=None, temperature=0):
    """Convenience wrapper for a single user turn (with optional system message)."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return chat(messages, temperature=temperature)

print("Setup complete.")

---
## 2. Zero-Shot Prompting

**What it is:** Ask the model to perform a task with *no* examples. You rely entirely on the model's pre-trained knowledge.

**When to use it:** Quick tasks where the model's general knowledge is sufficient; great as a baseline before trying more advanced techniques.

**Limitations:** May struggle with niche tasks or tasks that require a specific output format.

In [ ]:
# --- Zero-Shot: Sentiment Classification ---
zero_shot_prompt = """
Classify the sentiment of the following sentence as Positive, Negative, or Neutral.

Sentence: "The battery life of this phone is absolutely terrible."
Sentiment:"""

response = ask(zero_shot_prompt)
print("Zero-Shot Response:", response)

In [ ]:
# --- Zero-Shot: Translation ---
zero_shot_translation = "Translate the following English sentence to French: 'The weather is beautiful today.'"

response = ask(zero_shot_translation)
print("Zero-Shot Translation:", response)

---
## 3. Few-Shot Prompting

**What it is:** Provide a small number of input/output *examples* ("shots") before the actual task. The model learns the desired format and task from the examples.

**When to use it:** When the task has a specific output format, style, or domain-specific knowledge the model might not infer on its own.

**Tips:**
- Use 2–8 examples; more is not always better.
- Keep examples representative and diverse.
- Order matters: put the hardest or most representative example last.

In [ ]:
# --- Few-Shot: Sentiment Classification (same task, now with examples) ---
few_shot_prompt = """
Classify the sentiment of each sentence as Positive, Negative, or Neutral.

Sentence: "I love the new design of this app."
Sentiment: Positive

Sentence: "The customer support was completely unhelpful."
Sentiment: Negative

Sentence: "The package arrived on Tuesday."
Sentiment: Neutral

Sentence: "The battery life of this phone is absolutely terrible."
Sentiment:"""

response = ask(few_shot_prompt)
print("Few-Shot Response:", response)

In [ ]:
# --- Few-Shot: Custom entity extraction ---
few_shot_extraction = """
Extract the city and country from each sentence in JSON format.

Sentence: "I visited the Eiffel Tower in Paris, France last summer."
Output: {"city": "Paris", "country": "France"}

Sentence: "She grew up in São Paulo, Brazil."
Output: {"city": "São Paulo", "country": "Brazil"}

Sentence: "The conference will be held in Tokyo, Japan next month."
Output:"""

response = ask(few_shot_extraction)
print("Few-Shot Extraction:", response)

---
## 4. Chain-of-Thought (CoT) Prompting

**What it is:** Encourage the model to reason step-by-step before giving the final answer. The phrase *"Let's think step by step"* (zero-shot CoT) or in-context step-by-step examples (few-shot CoT) both work.

**Why it works:** Complex reasoning errors often stem from the model jumping to conclusions. Breaking the problem into intermediate steps mirrors how humans solve hard problems and significantly improves accuracy.

**When to use it:** Math word problems, logical reasoning, multi-step analysis, code debugging.

In [ ]:
# --- Without CoT ---
problem = """
A store sells apples for $0.50 each and oranges for $0.75 each.
Alice buys 4 apples and 3 oranges. She pays with a $5 bill.
How much change does she receive?"""

response_no_cot = ask(problem)
print("Without CoT:", response_no_cot)

In [ ]:
# --- Zero-Shot CoT: append "Let's think step by step." ---
cot_prompt = problem + "\n\nLet's think step by step."

response_cot = ask(cot_prompt)
print("Zero-Shot CoT:", response_cot)

In [ ]:
# --- Few-Shot CoT: include a solved example with explicit reasoning ---
few_shot_cot = """
Solve the following word problems step by step.

Problem: A baker made 24 cookies and packed them equally into 4 boxes.
She then gave 1 box to her neighbour. How many cookies does she have left?
Solution:
  Step 1: Cookies per box = 24 / 4 = 6
  Step 2: Cookies given away = 1 box × 6 cookies = 6
  Step 3: Cookies remaining = 24 - 6 = 18
Answer: 18 cookies

Problem: A store sells apples for $0.50 each and oranges for $0.75 each.
Alice buys 4 apples and 3 oranges. She pays with a $5 bill.
How much change does she receive?
Solution:"""

response_few_shot_cot = ask(few_shot_cot)
print("Few-Shot CoT:", response_few_shot_cot)

---
## 5. Role Prompting

**What it is:** Assign the model a specific persona, expert role, or character through the system message (or the start of the user message). The model then answers through that lens.

**Why it works:** Framing the model as an expert primes it to use specialised vocabulary, adopt the appropriate communication style, and draw on domain-specific knowledge.

**When to use it:** Domain-specific Q&A, tutoring, creative writing, code review, customer support.

In [ ]:
# --- Role: Senior Security Engineer ---
security_role = "You are a senior cybersecurity engineer with 15 years of experience in application security and penetration testing. Provide technically precise, actionable answers."

security_question = "What are the top three security risks when storing user passwords, and how should they be mitigated?"

response = ask(security_question, system=security_role)
print("Security Engineer Role:\n", response)

In [ ]:
# --- Same question, different role: 5th-grade teacher ---
teacher_role = "You are a friendly and patient 5th-grade teacher who explains complex topics using simple words, analogies, and relatable examples for 10-year-old students."

response_teacher = ask(security_question, system=teacher_role)
print("5th-Grade Teacher Role:\n", response_teacher)

---
## 6. Self-Consistency

**What it is:** Generate *multiple* independent reasoning paths (at a higher temperature) for the same question, then aggregate the answers by majority vote (or other aggregation).

**Why it works:** A single chain of thought can go astray. Sampling diverse reasoning paths and picking the most common answer dramatically reduces errors on complex reasoning tasks.

**When to use it:** Math and logical reasoning problems where you need high confidence in the final answer.

In [ ]:
from collections import Counter

reasoning_problem = """
If a train travels at 60 mph for 2.5 hours, then slows to 40 mph for the next 1.5 hours,
what is the total distance travelled?
Think step by step and end your answer with: "Final answer: X miles"
"""

NUM_SAMPLES = 5  # number of independent reasoning paths
answers = []

for i in range(NUM_SAMPLES):
    response = ask(reasoning_problem, temperature=0.7)
    # Extract the final answer line
    for line in response.split("\n"):
        if "final answer" in line.lower():
            answers.append(line.strip())
            print(f"Sample {i+1}: {line.strip()}")
            break

# Majority vote
vote_counts = Counter(answers)
winner = vote_counts.most_common(1)[0]
print(f"\nMajority vote: '{winner[0]}' ({winner[1]}/{NUM_SAMPLES} votes)")

---
## 7. Prompt Chaining

**What it is:** Break a complex task into a *sequence of smaller prompts*, where the output of each step feeds into the next.

**Why it works:** Each step is easier for the model and can be validated independently. It also allows you to insert logic, filtering, or human review between steps.

**When to use it:** Document processing pipelines, multi-step content generation, workflows with conditional branching.

In [ ]:
raw_article = """
Artificial intelligence is rapidly transforming the healthcare industry. 
Machine learning algorithms can now detect certain cancers with greater accuracy than human radiologists. 
AI-powered chatbots are helping patients triage symptoms before seeing a doctor, reducing emergency room congestion. 
Drug discovery timelines that once took over a decade are being compressed to just a few years thanks to AI simulation. 
However, concerns about data privacy, algorithmic bias, and the digital divide remain significant hurdles.
"""

# --- Step 1: Extract key points ---
step1_prompt = f"Extract the 3 most important key points from the following article as a numbered list:\n\n{raw_article}"
key_points = ask(step1_prompt)
print("Step 1 – Key Points:\n", key_points)

# --- Step 2: Generate a tweet thread from the key points ---
step2_prompt = f"""Based on the following key points, write a Twitter/X thread of 3 tweets (each ≤280 characters).
Number each tweet.

Key points:
{key_points}"""
tweet_thread = ask(step2_prompt)
print("\nStep 2 – Tweet Thread:\n", tweet_thread)

# --- Step 3: Suggest a compelling headline for the thread ---
step3_prompt = f"Write a single compelling headline (max 10 words) for this Twitter thread:\n\n{tweet_thread}"
headline = ask(step3_prompt)
print("\nStep 3 – Headline:\n", headline)

---
## 8. Tree of Thoughts (ToT)

**What it is:** An extension of CoT where the model *explores multiple reasoning branches* simultaneously and evaluates each path before committing to a final answer — mimicking a deliberate search process.

**Why it works:** Some problems have multiple plausible approaches. Exploring and evaluating branches allows backtracking from dead ends, leading to better solutions.

**When to use it:** Creative problem-solving, planning tasks, puzzles that require lookahead.

In [ ]:
# ToT via a single structured prompt that asks the model to simulate multiple experts
tot_prompt = """
Imagine three different experts are answering the following question.
All experts independently write their first reasoning step, then share it with the group.
The group evaluates each step, selects the most promising direction, and continues.
If any expert realises their approach is flawed, they drop out.
Continue until the best answer is reached and state it clearly.

Question: A farmer has 17 sheep. All but 9 die. How many sheep are left?
"""

response = ask(tot_prompt)
print("Tree of Thoughts Response:\n", response)

In [ ]:
# ToT: planning problem
planning_tot = """
Three experts in product strategy, engineering, and user research are collaborating.
They each propose a first step, discuss, select the best path, and iterate until they
reach a concrete action plan.

Problem: A mobile app has high user acquisition but a 70% churn rate in the first week.
Devise a prioritised action plan to reduce churn.
"""

response = ask(planning_tot)
print("ToT Planning Response:\n", response)

---
## 9. ReAct Prompting

**What it is:** The model interleaves **Re**asoning (thinking about what to do) and **Act**ing (calling a tool or taking an action), then observes the result and repeats until the task is complete.

**Why it works:** Grounding reasoning in real observations (tool outputs) prevents hallucination and allows the model to handle dynamic, multi-step tasks.

**When to use it:** Agent workflows with tool use (search, code execution, database queries).

> **Note:** The example below simulates the ReAct pattern using a structured prompt (no live tools). In production, each `Action` would actually call a function.

In [ ]:
react_prompt = """
You are an AI assistant that solves tasks by alternating between Thought, Action, and Observation steps.

Available actions:
  - Search[query]: search the web for information
  - Calculate[expression]: evaluate a mathematical expression
  - Finish[answer]: return the final answer

Use the following format strictly:
  Thought: <your reasoning>
  Action: <ActionName>[<input>]
  Observation: <result of the action>   ← You will be given this; simulate it if needed.
  ... (repeat Thought/Action/Observation as needed)
  Thought: I now know the final answer.
  Action: Finish[<answer>]

Task: What is the population of the capital city of Australia, and how does it compare (as a percentage) to Australia's total population of 26 million?

Begin!
Thought:"""

response = ask(react_prompt)
print("ReAct Response:\n", response)

---
## 10. Generated Knowledge Prompting

**What it is:** First ask the model to *generate relevant background knowledge or facts* about a topic, then feed that generated knowledge back into a second prompt to answer the actual question.

**Why it works:** Generating knowledge separately allows the model to surface relevant details it might otherwise skip when answering directly. The second prompt is then grounded in richer context.

**When to use it:** Common-sense reasoning tasks, factual Q&A, and any task where extra context clearly helps.

In [ ]:
question = "Is it healthier to drink coffee before or after a workout?"

# --- Step 1: Generate knowledge ---
knowledge_prompt = f"""
Generate 4–5 key scientific facts relevant to the following question.
Be concise and factual.

Question: {question}

Facts:"""

knowledge = ask(knowledge_prompt)
print("Generated Knowledge:\n", knowledge)

# --- Step 2: Use the knowledge to answer ---
answer_prompt = f"""
Using the following facts, answer the question clearly and concisely.

Facts:
{knowledge}

Question: {question}
Answer:"""

answer = ask(answer_prompt)
print("\nKnowledge-Grounded Answer:\n", answer)

---
## 11. Directional Stimulus Prompting

**What it is:** Include a *hint*, *keyword*, or *directive* in the prompt that nudges the model toward the desired type of answer — without giving the full answer away.

**Why it works:** Subtle steering signals act like a compass: they guide the generation direction without over-constraining the output.

**When to use it:** Summarisation with a specific angle, creative writing with a particular tone, or Q&A where you want the model to emphasise certain aspects.

In [ ]:
article = """
Remote work has fundamentally changed how companies operate. Employees report higher
satisfaction due to eliminated commutes and greater flexibility. Productivity metrics
are mixed: some studies show a 13% increase, while others point to collaboration
challenges and blurred work-life boundaries. Companies have reduced office costs
significantly, but some leaders argue that innovation suffers without in-person
spontaneity. Meanwhile, workers in rural areas now access job markets previously
unavailable to them.
"""

# Without stimulus
neutral_prompt = f"Summarise the following article in 2 sentences:\n\n{article}"
neutral_summary = ask(neutral_prompt)
print("Neutral Summary:\n", neutral_summary)

# With directional stimulus: focus on productivity
stimulus_prompt = f"""
Summarise the following article in 2 sentences.
Hint: focus on the **productivity and innovation** angle.

{article}"""
productivity_summary = ask(stimulus_prompt)
print("\nProductivity-Focused Summary:\n", productivity_summary)

# With directional stimulus: focus on workers
worker_stimulus_prompt = f"""
Summarise the following article in 2 sentences.
Hint: highlight the **impact on workers and work-life balance**.

{article}"""
worker_summary = ask(worker_stimulus_prompt)
print("\nWorker-Focused Summary:\n", worker_summary)

---
## Summary: Choosing the Right Technique

| Technique | Best For | Key Trade-off |
|-----------|----------|---------------|
| **Zero-Shot** | Quick tasks, baselines | Simple; may lack format/domain specificity |
| **Few-Shot** | Specific formats, niche domains | Requires good examples; uses more tokens |
| **Chain-of-Thought** | Math, logic, multi-step reasoning | Longer output; slower |
| **Role Prompting** | Domain expertise, tone control | Over-roleplay risk; needs clear persona |
| **Self-Consistency** | High-stakes reasoning answers | Multiple API calls; higher cost |
| **Prompt Chaining** | Complex multi-stage pipelines | More engineering; inter-step errors |
| **Tree of Thoughts** | Creative planning, hard puzzles | High token usage; complex to orchestrate |
| **ReAct** | Agentic tool-use workflows | Requires tool integration; longer loops |
| **Generated Knowledge** | Factual / common-sense reasoning | Two-step cost; generated facts may be wrong |
| **Directional Stimulus** | Summaries with specific angles | Subtle; works best for open-ended tasks |

### General Tips
- **Start simple** — zero-shot before few-shot before CoT.
- **Be explicit** — spell out the format, length, and constraints you expect.
- **Iterate** — small wording changes can have large effects; log your prompts.
- **Combine techniques** — e.g., few-shot + CoT, or role prompting + directional stimulus.
- **Evaluate** — define a metric (accuracy, BLEU, human preference) and measure it.